<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "516"
iso_country = "Namibia"
year = 2024
import_portfolio = "h6_ag6_by_sea_FM.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 13.400*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Namibia': 

Number of trade records present in dataset: 14679
Number of partner countries present in dataset: 131
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$3558.1m
Total weight within the dataset: 4082.1 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,516,Namibia,156,China,Foreign Import,Sea,690590,690590 | Ceramic chimney-pots,6905,"6905 | Roofing tiles, chimn",69,69 | Ceramic products,3735,"Non-refractory ceramic building bricks, floori...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,163.513,151.087,163.513,kg,172.85,172.85,588.466936,16004.0,9944.421484,5851.963238,['Container'],['Container'],19.844558,19.760026,19.765596,19.765596,19.822891,19.822891,20.855063,39.774410,23.319849,21.127568,21.213717,21.917431,20.374562,37.554136,24.043609,22.049204,21.575605,22.661651,19.953699,41.381085,30.213629,26.203949,24.254723,23.687386,19.537951,44.040070,35.005688,32.975408,25.651128,25.486042,19.129752,44.676320,41.522042,42.146933,28.044462,26.811393,18.713853,43.969176,42.672445,45.021109,32.893512,27.316049,18.322806,39.265102,39.929833,41.846933,31.852573,27.653859,17.925584,35.057548,36.094937,42.570338,34.723604,27.871740,3735
1,1,516,Namibia,826,United Kingdom,Foreign Import,Sea,690590,690590 | Ceramic chimney-pots,6905,"6905 | Roofing tiles, chimn",69,69 | Ceramic products,3735,"Non-refractory ceramic building bricks, floori...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,39.784,4.063,39.784,kg,71.00,71.00,143.178638,10180.0,6325.556780,905.684602,['Container'],['Container'],3.071262,3.058179,3.059041,3.059041,3.067908,3.067908,3.227654,6.155724,3.609118,3.269828,3.283161,3.392072,3.153288,5.812101,3.721132,3.412466,3.339169,3.507252,3.088153,6.404383,4.676041,4.055479,3.753805,3.666001,3.023809,6.815903,5.417688,5.103470,3.969921,3.944371,2.960634,6.914373,6.426198,6.522910,4.340328,4.149490,2.896267,6.804931,6.604241,6.967734,5.090795,4.227594,2.835746,6.076901,6.179778,6.476480,4.929693,4.279876,2.774270,5.425714,5.586267,6.588438,5.374031,4.313596,3735


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $243.3m (6.8%), ... 

                             cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                            
17 | Sugars and sugar con  71922749.4  159114281.0      2.0      3.9
10 | Cereals               61523644.6  207755251.5      1.7      5.1
11 | Products of the mill  28954902.3   53957071.9      0.8      1.3
02 | Meat and edible meat  20842484.6   10741603.0      0.6      0.3
03 | Fish and crustaceans  15369753.8    4997961.2      0.4      0.1
16 | Meat, fish, crustace  14190512.0    3009776.0      0.4      0.1
22 | Beverages, spirits a   9246161.9    8353012.8      0.3      0.2
19 | Preparations of cere   5959020.2    2599800.9      0.2      0.1
21 | Miscellaneous edible   3462109.5    1201025.0      0.1      0.0
04 | Dairy produce; birds   3147338.9    1071980.5      0.1      0.0
20 | Preparations of vege   2320598.7    1885778.1      0.1      0.0
15 

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $178.4m (5.0%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Food & Beverages  102836364.9  167598550.0      2.9      4.1
Agriculture        70523842.9  100340568.6      2.0      2.5
Fishing             5086968.5    2251623.8      0.1      0.1



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $144.2m (4.1%), ... 

                                 cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                               
SGR - Sugar                    68057266.4  152718395.7      1.9      3.7
OFD - Food products n.e.c.     34794327.5   13730406.1      1.0      0.3
OMT - Meat products n.e.c.     19836776.1    9729313.6      0.6      0.2
PCR - Processed rice            9098781.6   15804583.6      0.3      0.4
GRO - Cereal grains n.e.c.      3957042.8   16367650.3      0.1      0.4
MIL - Dairy products            3096982.1    1044552.6      0.1      0.0
CMT - Bovine meat prods         2626227.6    1340612.6      0.1      0.0
VOL - Vegetable oils and fats   1272990.8    1207608.0      0.0      0.0
OCR - Crops n.e.c.               911839.8     134665.6      0.0      0.0
OAP - Animal products n.e.c.     261215.7      61653.4      0.0      0.0
V_F - Vegetables, fruit, nuts    230115.8     485387.2     

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 6.8% 
GTAP 4.1% 
EORA 5.0%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46403/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,516,Namibia,156,China,Foreign Import,Sea,690590,690590 | Ceramic chimney-pots,6905,"6905 | Roofing tiles, chimn",69,69 | Ceramic products,3735,"Non-refractory ceramic building bricks, floori...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,163.513,151.087,163.513,kg,172.85,172.85,588.466936,16004.0,9944.421484,5851.963238,['Container'],['Container'],19.844558,19.760026,19.765596,19.765596,19.822891,19.822891,20.855063,39.774410,23.319849,21.127568,21.213717,21.917431,20.374562,37.554136,24.043609,22.049204,21.575605,22.661651,19.953699,41.381085,30.213629,26.203949,24.254723,23.687386,19.537951,44.040070,35.005688,32.975408,25.651128,25.486042,19.129752,44.676320,41.522042,42.146933,28.044462,26.811393,18.713853,43.969176,42.672445,45.021109,32.893512,27.316049,18.322806,39.265102,39.929833,41.846933,31.852573,27.653859,17.925584,35.057548,36.094937,42.570338,34.723604,27.871740,3735,NaN
1,1,516,Namibia,826,United Kingdom,Foreign Import,Sea,690590,690590 | Ceramic chimney-pots,6905,"6905 | Roofing tiles, chimn",69,69 | Ceramic products,3735,"Non-refractory ceramic building bricks, floori...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,39.784,4.063,39.784,kg,71.00,71.00,143.178638,10180.0,6325.556780,905.684602,['Container'],['Container'],3.071262,3.058179,3.059041,3.059041,3.067908,3.067908,3.227654,6.155724,3.609118,3.269828,3.283161,3.392072,3.153288,5.812101,3.721132,3.412466,3.339169,3.507252,3.088153,6.404383,4.676041,4.055479,3.753805,3.666001,3.023809,6.815903,5.417688,5.103470,3.969921,3.944371,2.960634,6.914373,6.426198,6.522910,4.340328,4.149490,2.896267,6.804931,6.604241,6.967734,5.090795,4.227594,2.835746,6.076901,6.179778,6.476480,4.929693,4.279876,2.774270,5.425714,5.586267,6.588438,5.374031,4.313596,3735,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $195.5m of $3558.1m (5.5%)
Volume: 436.2 of 4082.1 million kilograms (10.7%)
Value-to-Weight: 0.45 from 0.87 US$ per kg (+-48.6%)
Average distance to market (volume-weighted): 5284.3 from 6602.3 nm (+-20.0%)

            food_group  N_com    cifvalue   vol_kg_vtw  pct_val  pct_vol
0      Sugar and Honey     55  71925012.8  159114929.9      2.0      3.9
1  Unprocessed Cereals     15  52424756.4  191950482.6      1.5      4.7
2    Processed Cereals    309  47675450.5   72616816.6      1.3      1.8
3        Meat and Prep     57  18212884.5    9399560.9      0.5      0.2
4                Dairy     61   3107334.9    1055656.4      0.1      0.0
5        Fats and Oils     48   1485607.6    1163215.1      0.0      0.0
6           Vegetables    180    649701.2     826679.3      0.0      0.0
7               Fruits     62     68087.7      68878.1      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,2536020.6,4834153.5,2835197.1,2569085.4,2579545.8,2665066.6,2326435.4,5.430663e+06,5.046972e+06,5.123510e+06,3409318.7,3259683.8,2180133.7,4262180.7,4388147.2,5.174853e+06,4220958.1,3388647.0
1,Fats and Oils,3037333.1,5771040.0,3391579.3,3076392.2,3088803.4,3190843.6,2787934.1,6.488550e+06,6.028095e+06,6.123896e+06,4076225.6,3899236.4,2613676.0,5098062.2,5247565.0,6.184429e+06,5044074.0,4053838.4
2,Fruits,56856.7,107446.2,63371.2,57611.3,57834.0,59710.0,52230.5,1.209316e+05,1.122653e+05,1.140627e+05,76103.6,72882.1,48996.9,95495.3,98191.1,1.154264e+05,93913.1,75776.1
3,Meat and Prep,18359415.4,35014754.2,20529248.9,18599310.3,18675150.2,19294653.6,16840565.6,3.933007e+07,3.655325e+07,3.710337e+07,24688485.2,23602973.3,15780495.9,30862341.4,31775589.7,3.747610e+07,30568359.6,24536432.1
4,Processed Cereals,168817001.2,318598120.6,188128125.8,170863895.2,171548416.3,177243553.9,155130061.0,3.590616e+08,3.340514e+08,3.394386e+08,226646922.5,216053613.5,145576277.3,282353549.3,290450696.3,3.419319e+08,279449949.1,224441415.1
5,Sugar and Honey,117575564.5,214825622.7,129693031.1,118662939.9,119112631.8,123083685.8,108626798.2,2.444603e+08,2.282559e+08,2.326780e+08,157050097.8,148425330.2,102379691.8,193547155.3,198573920.6,2.324915e+08,191175687.1,153848990.7
6,Unprocessed Cereals,508540484.5,966687237.3,568038715.2,515032514.3,517121329.3,534282222.1,466733803.4,1.086968e+09,1.010573e+09,1.026108e+09,683533819.7,652850947.6,437555108.0,853497643.3,878507284.5,1.035521e+09,845186452.6,678513955.6
7,Vegetables,1884179.0,3426446.3,2070493.8,1903847.1,1910642.3,1971008.5,1742681.6,3.899825e+06,3.611747e+06,3.702334e+06,2475089.5,2379904.2,1642126.8,3105561.0,3187715.1,3.725640e+06,3037659.4,2475755.2


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.74,0.10,0.01,0.01,0.04,1.00,0.88,0.90,0.35,0.30,0.67,0.71,0.96,0.66,0.39
1,Fats and Oils,1.84,0.24,0.03,0.03,0.10,2.49,2.18,2.25,0.87,0.75,1.67,1.77,2.40,1.64,0.97
2,Fruits,0.74,0.10,0.01,0.01,0.04,1.01,0.88,0.91,0.35,0.30,0.68,0.72,0.98,0.66,0.39
3,Meat and Prep,0.91,0.12,0.01,0.02,0.05,1.23,1.08,1.11,0.43,0.37,0.83,0.88,1.19,0.81,0.48
4,Processed Cereals,3.14,0.41,0.04,0.06,0.18,4.28,3.75,3.87,1.50,1.28,2.87,3.04,4.12,2.81,1.65
5,Sugar and Honey,1.35,0.17,0.02,0.02,0.08,1.89,1.66,1.72,0.67,0.55,1.27,1.34,1.81,1.23,0.72
6,Unprocessed Cereals,8.74,1.13,0.12,0.16,0.49,11.83,10.37,10.67,4.14,3.55,7.93,8.41,11.41,7.78,4.60
7,Vegetables,2.37,0.29,0.03,0.04,0.13,3.32,2.88,3.02,1.13,0.98,2.25,2.38,3.21,2.15,1.28
